# Phase 2 (a): User-Based Collaborative Filtering — KNN

This notebook trains a **User-Based KNN** recommender on CiaoDVD using the
[Surprise](https://surprise.readthedocs.io/) library, with a small grid search
over `k` (number of neighbors) using 3-fold cross-validation.

We use a fixed `random_state=42` for the train/test split so that this notebook
and `03_svd_model.ipynb` are evaluated on the **exact same** test set.


In [ ]:
import sys
from pathlib import Path

# Allow `from src.xxx import yyy` when this notebook lives in /notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split

from src.cf_model import tune_user_knn, fit_user_knn

RANDOM_STATE = 42  # MUST match the SVD notebook


## 1. Load cleaned data

(Run `01_eda.ipynb` first to produce this file.)

In [ ]:
ratings = pd.read_csv("data/processed/ratings_clean.csv")
print(f"Loaded {len(ratings):,} ratings")
ratings.head()


## 2. Build the Surprise Dataset and 80/20 split

In [ ]:
# Surprise expects (user, item, rating) — we drop movie_categoryId for modeling.
model_data = ratings[["userId", "movieId", "movieRating"]]

reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(model_data, reader)

trainset, testset = surprise_split(data, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train interactions: {trainset.n_ratings:,}")
print(f"Test  interactions: {len(testset):,}")


## 3. Hyperparameter tuning (3-fold CV)

We grid-search `k` (number of neighbors) with cosine similarity. KNNBasic with
`user_based=True` finds users with similar rating patterns and predicts a
rating as a weighted average of their ratings.

In [ ]:
best_params = tune_user_knn(data, cv=3)
print("Best KNN params:", best_params)


## 4. Train best KNN on the trainset and evaluate on the held-out testset

In [ ]:
knn = fit_user_knn(trainset, best_params)
knn_predictions = knn.test(testset)

rmse_knn = accuracy.rmse(knn_predictions)
mae_knn  = accuracy.mae(knn_predictions)


## 5. Append metrics to results/metrics.csv

In [ ]:
import os

row = pd.DataFrame([{
    "Model": "User-Based KNN",
    "RMSE":  rmse_knn,
    "MAE":   mae_knn,
}])

path = "results/metrics.csv"
if os.path.exists(path):
    existing = pd.read_csv(path)
    existing = existing[existing["Model"] != "User-Based KNN"]  # avoid duplicates
    out = pd.concat([existing, row], ignore_index=True)
else:
    out = row
out.to_csv(path, index=False)
out


**Note on KNN performance.** On a >99.97% sparse rating matrix, most
pairs of users share very few rated movies. With so little overlap, the
similarity estimates are noisy, which limits how much KNN can learn. We
expect SVD to do better — see `03_svd_model.ipynb`.